# SKU MatchOps — Resumable Multi-Precision Evaluation (Colab GPU / CPU)

Evaluates all domain × embedding × reranker × precision × mode × BT-head × ablation combinations with leak-free `SellerCategory` inputs, GLiNER entity extraction, Template Tag Enrichment, Rules Engine, and Drive-backed checkpoints.

### Matrix Covered
- **Domains:** `market` (`7,421` train / `1,702` test), `food` (`61,017` train / `15,259` test)
- **Embeddings (3):** `BAAI/bge-m3`, `jinaai/jina-embeddings-v5-text-small`, `sentence-transformers/all-mpnet-base-v2`
- **Rerankers (4):** `BAAI/bge-reranker-v2-m3`, `jinaai/jina-reranker-v2-base-multilingual`, `jinaai/jina-reranker-v3.5`, `cross-encoder/ms-marco-MiniLM-L6-v2`
- **Precisions (3):** `fp32`, `fp16` (CUDA GPU / PyTorch FP16), `int8` (ONNX CPU)
- **Modes & BT Heads:** `matcher`, `classifier`, `pipeline` × `arcface`, `logreg` × Templates (`on/off`) × Rules (`on/off`)

### Recommended Workflow
1. Select **Runtime → Change runtime type → L4 / A100 / T4 GPU** (and High-RAM if available).
2. **Cell 1:** Mount Google Drive, clone/pull repo, install GPU ONNX Runtime & dependencies.
3. **Cell 2:** Start native local Qdrant server binary on fast Colab SSD (`/content/qdrant_storage`).
4. **Cell 3:** Select preset (`smoke_100`, `market_all`, `food_gpu_fp16_fp32`, `food_int8_only`, `food_all`, or `custom`) and verify data fixtures.
5. **Cell 4:** Run evaluation (saves Stage A NER, Stage B embeddings/retrieval, Stage C rerank pair scores every 5,000 pairs, and `summary.json` directly to Drive).
6. **Cell 5:** Inspect leaderboards, ablation tables, Markdown report, and download results zip.

In [ ]:
# @title 1. Mount Drive, Check Hardware, Pull Code & Install Dependencies
import os
import subprocess
import sys
from pathlib import Path

# --- Repository & Drive Configuration ---
REPO_URL = "https://github.com/muaazl/sku-matchops.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
USE_COLAB_GH_TOKEN_SECRET = False  # @param {type:"boolean"}
DRIVE_EVAL_ROOT = "/content/drive/MyDrive/sku_matchops_eval"  # @param {type:"string"}
PERSIST_MODELS_ON_DRIVE = False  # @param {type:"boolean"}

from google.colab import drive

drive.mount("/content/drive")

drive_root = Path(DRIVE_EVAL_ROOT)
for sub in ("runs_smoke", "runs_full", "eval_data", "models"):
    (drive_root / sub).mkdir(parents=True, exist_ok=True)

# Optional HF_TOKEN from Colab secrets avoids HuggingFace rate-limits
try:
    from google.colab import userdata
    hf_tok = userdata.get("HF_TOKEN")
    if hf_tok:
        os.environ["HF_TOKEN"] = hf_tok
except Exception:
    pass

clone_url = REPO_URL.strip()
if USE_COLAB_GH_TOKEN_SECRET:
    from google.colab import userdata
    gh_tok = userdata.get("GH_TOKEN")
    if gh_tok and clone_url.startswith("https://"):
        clone_url = clone_url.replace("https://", f"https://{gh_tok}@", 1)

repo_dir = Path("/content/sku-matchops")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, clone_url, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=False)

os.chdir(str(repo_dir))

# Install GPU ONNX Runtime (remove CPU onnxruntime first so CUDAExecutionProvider is preserved)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime"], check=False)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "onnxruntime-gpu", "onnx", "onnxscript",
    "qdrant-client>=1.12",
    "transformers>=4.46,<5.0",
    "sentence-transformers>=3.0",
    "FlagEmbedding>=1.2.9",
    "gliner>=0.2",
    "optimum>=1.18",
    "scikit-learn>=1.5",
    "polars", "pyarrow", "joblib", "quantulum3", "rapidfuzz",
    "rich", "tqdm", "tabulate", "pyyaml", "einops", "peft", "accelerate",
], check=True)

# Verify Hardware & Providers
import torch

import onnxruntime as ort  # isort: skip

if hasattr(ort, "preload_dlls"):
    try:
        ort.preload_dlls()
    except Exception as e:
        print("ort.preload_dlls note:", e)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-ONLY"
vram_gb = (torch.cuda.get_device_properties(0).total_memory / (1024**3)) if torch.cuda.is_available() else 0.0
cpu_cnt = os.cpu_count() or 2
print(f"Device: {gpu_name} ({vram_gb:.1f} GB VRAM) | CPU cores: {cpu_cnt}")
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
print(f"ORT version: {ort.__version__} | Providers: {ort.get_available_providers()}")
if torch.cuda.is_available() and "CUDAExecutionProvider" not in ort.get_available_providers():
    raise RuntimeError("CUDAExecutionProvider missing! Re-run: !pip uninstall -y onnxruntime && pip install -q --force-reinstall onnxruntime-gpu")

MODELS_ROOT = str(drive_root / "models") if PERSIST_MODELS_ON_DRIVE else "/content/eval_models"
Path(MODELS_ROOT).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(drive_root / "hf_cache") if PERSIST_MODELS_ON_DRIVE else "/content/hf_cache"
print(f"Drive root: {drive_root}")
print(f"Models root: {MODELS_ROOT} (persist_on_drive={PERSIST_MODELS_ON_DRIVE})")

In [ ]:
# @title 2. Start Native Linux Qdrant Server (Fast Hybrid Dense + Sparse + BM25)
import json
import subprocess
import time
import urllib.request
from pathlib import Path

QDRANT_URL = "http://127.0.0.1:6333"
QDRANT_STORAGE = "/content/qdrant_storage"
QDRANT_FALLBACK_TAG = "v1.17.0"

def _qdrant_ready(url: str) -> bool:
    try:
        with urllib.request.urlopen(f"{url}/readyz", timeout=2) as resp:
            return resp.status == 200
    except Exception:
        return False

if _qdrant_ready(QDRANT_URL):
    print(f"Qdrant server already running at {QDRANT_URL}")
else:
    qdrant_bin = Path("/usr/local/bin/qdrant")
    need_dl = not qdrant_bin.exists()
    if not need_dl:
        ver_chk = subprocess.run([str(qdrant_bin), "--version"], capture_output=True, text=True)
        if ver_chk.returncode != 0:
            print(f"Replacing incompatible qdrant binary ({ver_chk.stderr.strip()})...")
            qdrant_bin.unlink(missing_ok=True)
            need_dl = True

    if need_dl:
        dl_url = f"https://github.com/qdrant/qdrant/releases/download/{QDRANT_FALLBACK_TAG}/qdrant-x86_64-unknown-linux-musl.tar.gz"
        try:
            with urllib.request.urlopen("https://api.github.com/repos/qdrant/qdrant/releases/latest", timeout=10) as r:
                rel = json.loads(r.read().decode("utf-8"))
            assets = {a.get("name"): a["browser_download_url"] for a in rel.get("assets", [])}
            if "qdrant-x86_64-unknown-linux-musl.tar.gz" in assets:
                dl_url = assets["qdrant-x86_64-unknown-linux-musl.tar.gz"]
            elif "qdrant-x86_64-unknown-linux-gnu.tar.gz" in assets:
                dl_url = assets["qdrant-x86_64-unknown-linux-gnu.tar.gz"]
        except Exception as err:
            print(f"Using fallback Qdrant release {QDRANT_FALLBACK_TAG} ({err})")

        tgz = Path("/content/qdrant.tar.gz")
        print(f"Downloading Qdrant static binary from {dl_url}...")
        subprocess.run(["curl", "-fSL", dl_url, "-o", str(tgz)], check=True)
        subprocess.run(["tar", "-xzf", str(tgz), "-C", "/usr/local/bin", "qdrant"], check=True)
        qdrant_bin.chmod(0o755)
        tgz.unlink(missing_ok=True)
        subprocess.run([str(qdrant_bin), "--version"], check=True)

    Path(QDRANT_STORAGE).mkdir(parents=True, exist_ok=True)
    cfg_path = Path("/content/qdrant_config.yaml")
    cfg_path.write_text(
        "log_level: WARN\n"
        "storage:\n"
        f"  storage_path: {QDRANT_STORAGE}\n"
        "service:\n"
        "  host: 127.0.0.1\n"
        "  http_port: 6333\n"
        "  grpc_port: 6334\n"
        "  enable_cors: true\n",
        encoding="utf-8",
    )
    log_file = open("/content/qdrant_server.log", "ab")
    subprocess.Popen([str(qdrant_bin), "--config-path", str(cfg_path)], stdout=log_file, stderr=log_file)

    for _ in range(30):
        if _qdrant_ready(QDRANT_URL):
            break
        time.sleep(1)
    if not _qdrant_ready(QDRANT_URL):
        log_tail = Path("/content/qdrant_server.log").read_text(encoding="utf-8", errors="ignore")[-2000:]
        raise RuntimeError(f"Qdrant server failed to start; log tail:\n{log_tail}")
    print(f"Qdrant server ready at {QDRANT_URL} (storage={QDRANT_STORAGE})")

In [ ]:
# @title 3. Configure Preset, Prepare Local Eval Data & Auto-Tune Batch Sizes
import shutil
import zipfile
from pathlib import Path

import torch

# --- Run Configuration ---
PRESET = "smoke_100"  # @param ["smoke_100", "market_all", "food_gpu_fp16_fp32", "food_int8_only", "food_all", "full_all", "custom"]

# Used only when PRESET == "custom" (or override below):
CUSTOM_DOMAINS = "market,food"  # @param {type:"string"}
CUSTOM_EMBEDDINGS = "all"  # @param {type:"string"}
CUSTOM_RERANKERS = "all"  # @param {type:"string"}
CUSTOM_PRECISIONS = "all"  # @param {type:"string"}
MODES = "matcher,classifier,pipeline"  # @param {type:"string"}
HEADS = "arcface,logreg"  # @param {type:"string"}
ARCFACE_EPOCHS = 25  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}
NO_RESUME = False  # @param {type:"boolean"}

# Optional Drive zip extraction if you uploaded eval_data.zip to DRIVE_EVAL_ROOT/eval_data.zip
drive_eval_dir = Path(DRIVE_EVAL_ROOT) / "eval_data"
drive_eval_zip = Path(DRIVE_EVAL_ROOT) / "eval_data.zip"
if drive_eval_zip.exists() and not (drive_eval_dir / "train_market.csv").exists():
    print(f"Extracting {drive_eval_zip} -> {drive_eval_dir}...")
    with zipfile.ZipFile(drive_eval_zip, "r") as zf:
        zf.extractall(drive_eval_dir)

# Copy evaluation CSVs + fixtures to local Colab SSD (/content/eval_data) for fast pandas reads
repo_eval_dir = Path("/content/sku-matchops/evaluation")
local_eval_dir = Path("/content/eval_data")
src_eval = drive_eval_dir if (drive_eval_dir / "train_market.csv").exists() else repo_eval_dir
if not (src_eval / "train_market.csv").exists():
    raise FileNotFoundError(
        f"Missing evaluation splits! Put train_*.csv, test_*.csv, and fixtures/ inside either "
        f"{drive_eval_dir} or {repo_eval_dir} (via git repo)."
    )

local_eval_dir.mkdir(parents=True, exist_ok=True)
for fname in ("train_market.csv", "test_market.csv", "train_food.csv", "test_food.csv"):
    src_f = src_eval / fname
    if not src_f.exists():
        raise FileNotFoundError(f"Missing required split file: {src_f}")
    shutil.copy2(src_f, local_eval_dir / fname)

for rel in ("fixtures/rules_snapshot.json", "fixtures/staged_sheets"):
    src_p = src_eval / rel
    dst_p = local_eval_dir / rel
    if not src_p.exists():
        raise FileNotFoundError(f"Missing required fixture: {src_p}")
    if src_p.is_dir():
        shutil.copytree(src_p, dst_p, dirs_exist_ok=True)
    else:
        dst_p.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_p, dst_p)

PRESETS = {
    "smoke_100": {"domains": "market", "embeddings": "sentence-transformers/all-mpnet-base-v2", "rerankers": "cross-encoder/ms-marco-MiniLM-L6-v2", "precisions": "fp16,int8", "max_test_rows": 100, "stratified": True, "work_subdir": "runs_smoke"},
    "market_all": {"domains": "market", "embeddings": "all", "rerankers": "all", "precisions": "all", "max_test_rows": 0, "stratified": False, "work_subdir": "runs_full"},
    "food_gpu_fp16_fp32": {"domains": "food", "embeddings": "all", "rerankers": "all", "precisions": "fp16,fp32", "max_test_rows": 0, "stratified": False, "work_subdir": "runs_full"},
    "food_int8_only": {"domains": "food", "embeddings": "all", "rerankers": "all", "precisions": "int8", "max_test_rows": 0, "stratified": False, "work_subdir": "runs_full"},
    "food_all": {"domains": "food", "embeddings": "all", "rerankers": "all", "precisions": "all", "max_test_rows": 0, "stratified": False, "work_subdir": "runs_full"},
    "full_all": {"domains": "market,food", "embeddings": "all", "rerankers": "all", "precisions": "all", "max_test_rows": 0, "stratified": False, "work_subdir": "runs_full"},
    "custom": {"domains": CUSTOM_DOMAINS, "embeddings": CUSTOM_EMBEDDINGS, "rerankers": CUSTOM_RERANKERS, "precisions": CUSTOM_PRECISIONS, "max_test_rows": 0, "stratified": False, "work_subdir": "runs_full"},
}
CFG = PRESETS[PRESET]
WORK_DIR = str(Path(DRIVE_EVAL_ROOT) / CFG["work_subdir"])
EVAL_DATA_DIR = str(local_eval_dir)

vram_gb = (torch.cuda.get_device_properties(0).total_memory / (1024**3)) if torch.cuda.is_available() else 0.0
if vram_gb >= 35:
    EMBED_BS, RERANK_BS, ARCFACE_BS = 256, 128, 128
elif vram_gb >= 20:
    EMBED_BS, RERANK_BS, ARCFACE_BS = 192, 96, 128
elif vram_gb >= 12:
    EMBED_BS, RERANK_BS, ARCFACE_BS = 128, 64, 64
else:
    EMBED_BS, RERANK_BS, ARCFACE_BS = 64, 32, 64

print(f"Preset={PRESET} -> work_dir={WORK_DIR}")
print(f"Eval data staged at: {EVAL_DATA_DIR} (source={src_eval})")
print(f"Batch sizes (VRAM={vram_gb:.1f}GB): embed={EMBED_BS}, rerank={RERANK_BS}, arcface={ARCFACE_BS}")

In [ ]:
# @title 4. Run Resumable Evaluation
import json
import subprocess
import sys
from pathlib import Path

reg_file = Path(WORK_DIR) / "checkpoint_registry.json"
if reg_file.exists():
    prev_reg = json.loads(reg_file.read_text(encoding="utf-8"))
    print(f"Resuming from existing checkpoint_registry.json: {len(prev_reg.get('stage_c', {}))} Stage-C combos done.")

cmd = [
    sys.executable, "-u", "-m", "evaluation.run_eval",
    "--domains", CFG["domains"],
    "--embeddings", CFG["embeddings"],
    "--rerankers", CFG["rerankers"],
    "--precisions", CFG["precisions"],
    "--modes", MODES,
    "--heads", HEADS,
    "--work-dir", WORK_DIR,
    "--eval-dir", EVAL_DATA_DIR,
    "--models-root", MODELS_ROOT,
    "--qdrant-url", QDRANT_URL,
    "--device", "auto",
    "--epochs", str(ARCFACE_EPOCHS),
    "--embed-batch-size", str(EMBED_BS),
    "--rerank-batch-size", str(RERANK_BS),
    "--arcface-batch-size", str(ARCFACE_BS),
    "--max-test-rows", str(CFG["max_test_rows"]),
    "--seed", str(SEED),
]
if CFG["stratified"]:
    cmd.append("--stratified-slice")
if NO_RESUME:
    cmd.append("--no-resume")

print("Executing:", " ".join(cmd))
proc = subprocess.Popen(
    cmd,
    cwd="/content/sku-matchops",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"run_eval exited with code {ret}")
print("\nEvaluation completed cleanly.")

In [ ]:
# @title 5. Inspect Reports, Leaderboards & Archive Results
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

if "/content/sku-matchops" not in sys.path:
    sys.path.insert(0, "/content/sku-matchops")
from evaluation.report import generate_reports

paths = generate_reports(WORK_DIR)
for name, path in paths.items():
    print(f"{name}: {path}")

lb_path = paths.get("leaderboard.csv")
if lb_path and Path(lb_path).exists() and Path(lb_path).stat().st_size > 0:
    print("\n=== End-to-End Leaderboard (Top 20) ===")
    display(pd.read_csv(lb_path).head(20))

ret_path = paths.get("retrieval_and_embedding.csv")
if ret_path and Path(ret_path).exists() and Path(ret_path).stat().st_size > 0:
    print("\n=== Standalone Retrieval & Embeddings ===")
    display(pd.read_csv(ret_path).head(25))

head_path = paths.get("bt_heads_standalone.csv")
if head_path and Path(head_path).exists() and Path(head_path).stat().st_size > 0:
    print("\n=== Standalone Classifier Heads (ArcFace vs LogReg) ===")
    display(pd.read_csv(head_path))

md_path = paths.get("report.md")
if md_path and Path(md_path).exists():
    display(Markdown(Path(md_path).read_text(encoding="utf-8")))

zip_out = Path(WORK_DIR) / "matchops_eval_results.zip"
with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    wdir = Path(WORK_DIR)
    for pattern in ("checkpoint_registry.json", "reports/*", "results/*/*"):
        for f in wdir.glob(pattern):
            if f.is_file():
                zf.write(f, arcname=f.relative_to(wdir))
print(f"\nSaved portable results bundle: {zip_out}")